In [2]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/processed/ethiopia_fi_enriched.csv', parse_dates=['observation_date'])

events = df[df['record_type'] == 'event'].copy()
links = df[df['record_type'] == 'impact_link'].copy()

# Join impact_link to its parent event
merged = links.merge(
    events[['record_id', 'observation_date', 'category', 'original_text']],
    left_on='parent_id', right_on='record_id',
    suffixes=('_link', '_event')
)
merged[['record_id_event', 'observation_date_event', 'category_event', 'related_indicator',
        'impact_direction', 'impact_magnitude', 'lag_months', 'confidence']]

,record_id_event,observation_date_event,category_event,related_indicator,impact_direction,impact_magnitude,lag_months,confidence
0,EVT_0001,2021-05-17,product_launch,ACC_OWNERSHIP,increase,high,12.0,medium
1,EVT_0001,2021-05-17,product_launch,USG_TELEBIRR_USERS,increase,high,3.0,high
2,EVT_0001,2021-05-17,product_launch,USG_P2P_COUNT,increase,high,6.0,medium
3,EVT_0002,2022-08-01,market_entry,ACC_4G_COV,increase,medium,12.0,medium
4,EVT_0002,2022-08-01,market_entry,AFF_DATA_INCOME,decrease,medium,12.0,medium
5,EVT_0003,2023-08-01,product_launch,USG_MPESA_USERS,increase,high,3.0,high
6,EVT_0003,2023-08-01,product_launch,ACC_MM_ACCOUNT,increase,medium,6.0,medium
7,EVT_0004,2024-01-01,infrastructure,ACC_OWNERSHIP,increase,medium,24.0,medium
8,EVT_0004,2024-01-01,infrastructure,GEN_GAP_ACC,decrease,medium,24.0,medium
9,EVT_0005,2024-07-29,policy,AFF_DATA_INCOME,increase,high,3.0,high


In [3]:
# Map qualitative magnitude to an assumed percentage-point effect.
# These are assumptions — document them in your methodology writeup.
MAGNITUDE_MAP = {
    'high': 5.0,
    'medium': 2.5,
    'low': 1.0,
}

merged['magnitude_numeric'] = merged['impact_magnitude'].str.lower().map(MAGNITUDE_MAP)

merged['signed_impact'] = merged.apply(
    lambda r: r['magnitude_numeric'] if r['impact_direction'] == 'increase' else -r['magnitude_numeric'],
    axis=1
)

merged[['record_id_event', 'related_indicator', 'impact_direction',
        'impact_magnitude', 'magnitude_numeric', 'signed_impact', 'lag_months', 'confidence']]

,record_id_event,related_indicator,impact_direction,impact_magnitude,magnitude_numeric,signed_impact,lag_months,confidence
0,EVT_0001,ACC_OWNERSHIP,increase,high,5.0,5.0,12.0,medium
1,EVT_0001,USG_TELEBIRR_USERS,increase,high,5.0,5.0,3.0,high
2,EVT_0001,USG_P2P_COUNT,increase,high,5.0,5.0,6.0,medium
3,EVT_0002,ACC_4G_COV,increase,medium,2.5,2.5,12.0,medium
4,EVT_0002,AFF_DATA_INCOME,decrease,medium,2.5,-2.5,12.0,medium
5,EVT_0003,USG_MPESA_USERS,increase,high,5.0,5.0,3.0,high
6,EVT_0003,ACC_MM_ACCOUNT,increase,medium,2.5,2.5,6.0,medium
7,EVT_0004,ACC_OWNERSHIP,increase,medium,2.5,2.5,24.0,medium
8,EVT_0004,GEN_GAP_ACC,decrease,medium,2.5,-2.5,24.0,medium
9,EVT_0005,AFF_DATA_INCOME,increase,high,5.0,5.0,3.0,high
